# Performance & Benchmarking Tests

## Success Criteria: SC-01 Performance Parity
**Target**: Read/write latency within 10-15% of native tables

## Test Matrix
| Test | Approach | Success Criteria |
|------|----------|------------------|
| Large-Scale Scans | Full scans of 1M+ rows | Latency within 10% of native |
| Point Lookups | Search Optimization enabled | Sub-second response |
| Complex Joins | Multi-way joins | Execution plan efficiency |
| Aggregations | GROUP BY, window functions | Performance parity |
| Clustering | Auto-clustering depth/overlap | Convergence verification |
| **External vs Internal** | Compare storage backends | Within 5% of each other |

## Table Types Under Test
| Schema | Storage | Description |
|--------|---------|-------------|
| NATIVE_BASELINE | Snowflake Native | Standard Snowflake tables |
| TESTS | Iceberg (Snowflake-managed) | Iceberg v3 with auto paths |
| EXTERNAL_ICEBERG | Iceberg (Explicit paths) | Iceberg v3 with BASE_LOCATION |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE WAREHOUSE COMPUTE_WH;

---
## Test 1: Large-Scale Scan - 3-Way Comparison
Compare Native, Internal Iceberg, and External Iceberg tables.

In [ ]:
-- Verify all baseline data exists
SELECT 'Native' AS table_type, 'NATIVE_BASELINE.EVENTS' AS table_name, COUNT(*) AS row_count FROM ICEBERG_POC.NATIVE_BASELINE.EVENTS
UNION ALL
SELECT 'Iceberg (Internal)', 'TESTS.EVENTS', COUNT(*) FROM ICEBERG_POC.TESTS.EVENTS
UNION ALL
SELECT 'Iceberg (External)', 'EXTERNAL_ICEBERG.EVENTS', COUNT(*) FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS;

In [ ]:
-- Native table full scan
SELECT 
    'NATIVE' AS source,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    AVG(amount) AS avg_amount,
    SUM(amount) AS total_amount
FROM ICEBERG_POC.NATIVE_BASELINE.EVENTS;

In [ ]:
-- Internal Iceberg table full scan
SELECT 
    'ICEBERG_INTERNAL' AS source,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    AVG(amount) AS avg_amount,
    SUM(amount) AS total_amount
FROM ICEBERG_POC.TESTS.EVENTS;

In [ ]:
-- External Iceberg table full scan
SELECT 
    'ICEBERG_EXTERNAL' AS source,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    AVG(amount) AS avg_amount,
    SUM(amount) AS total_amount
FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS;

---
## Test 2: External Iceberg - Multi-Table Scan
Test all external Iceberg tables.

In [ ]:
-- External CUSTOMERS scan
SELECT 
    customer_tier,
    COUNT(*) AS customer_count,
    AVG(lifetime_value) AS avg_ltv,
    SUM(lifetime_value) AS total_ltv
FROM ICEBERG_POC.EXTERNAL_ICEBERG.CUSTOMERS
GROUP BY 1
ORDER BY customer_count DESC;

In [ ]:
-- External ORDERS scan
SELECT 
    order_status,
    COUNT(*) AS order_count,
    AVG(total_amount) AS avg_order_value,
    SUM(total_amount) AS total_revenue
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ORDERS
GROUP BY 1
ORDER BY order_count DESC;

In [ ]:
-- External TRANSACTIONS scan with nanosecond precision
SELECT 
    transaction_type,
    currency,
    COUNT(*) AS txn_count,
    AVG(amount) AS avg_amount,
    MIN(transaction_ts) AS earliest_txn,
    MAX(transaction_ts) AS latest_txn
FROM ICEBERG_POC.EXTERNAL_ICEBERG.TRANSACTIONS
GROUP BY 1, 2
ORDER BY txn_count DESC
LIMIT 20;

---
## Test 3: Point Lookups with Search Optimization

In [ ]:
-- Enable Search Optimization on Internal Iceberg table
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.EVENTS 
  ADD SEARCH OPTIMIZATION ON EQUALITY(customer_id, event_id);

In [ ]:
-- Enable Search Optimization on External Iceberg table
ALTER ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS 
  ADD SEARCH OPTIMIZATION ON EQUALITY(customer_id, event_id);

In [ ]:
-- Point lookup: Internal Iceberg
SELECT * FROM ICEBERG_POC.TESTS.EVENTS WHERE customer_id = 12345;

In [ ]:
-- Point lookup: External Iceberg
SELECT * FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS WHERE customer_id = 12345;

---
## Test 4: Auto-Clustering Comparison

In [ ]:
-- Enable clustering on Internal Iceberg
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.EVENTS CLUSTER BY (region, event_type);

-- Enable clustering on External Iceberg
ALTER ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS CLUSTER BY (region, event_type);

In [ ]:
-- Check clustering info for both
SELECT 'Internal' AS table_type, SYSTEM$CLUSTERING_INFORMATION('ICEBERG_POC.TESTS.EVENTS') AS clustering_info
UNION ALL
SELECT 'External', SYSTEM$CLUSTERING_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS');

---
## Test 5: Complex Joins - External Tables

In [ ]:
-- Multi-way join on External Iceberg tables
SELECT 
    c.customer_tier,
    c.region,
    o.order_status,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.total_amount) AS total_revenue,
    AVG(c.lifetime_value) AS avg_customer_ltv
FROM ICEBERG_POC.EXTERNAL_ICEBERG.CUSTOMERS c
JOIN ICEBERG_POC.EXTERNAL_ICEBERG.ORDERS o ON c.customer_id = o.customer_id
GROUP BY 1, 2, 3
ORDER BY total_revenue DESC
LIMIT 20;

In [ ]:
-- Join External tables with Transactions
SELECT 
    c.customer_tier,
    t.transaction_type,
    t.currency,
    COUNT(*) AS txn_count,
    SUM(t.amount) AS total_amount,
    AVG(t.metadata:risk_score::INT) AS avg_risk_score
FROM ICEBERG_POC.EXTERNAL_ICEBERG.CUSTOMERS c
JOIN ICEBERG_POC.EXTERNAL_ICEBERG.TRANSACTIONS t ON c.customer_id = t.customer_id
GROUP BY 1, 2, 3
ORDER BY txn_count DESC
LIMIT 20;

---
## Test 6: VARIANT Query Performance - External Tables

In [ ]:
-- Query VARIANT in External ORDERS
SELECT 
    shipping_address:city::STRING AS city,
    COUNT(*) AS order_count,
    AVG(total_amount) AS avg_order_value
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ORDERS
GROUP BY 1
ORDER BY order_count DESC
LIMIT 10;

In [ ]:
-- Query VARIANT in External TRANSACTIONS
SELECT 
    metadata:channel::STRING AS channel,
    metadata:payment_method::STRING AS payment_method,
    COUNT(*) AS txn_count,
    AVG(amount) AS avg_amount,
    AVG(metadata:risk_score::INT) AS avg_risk
FROM ICEBERG_POC.EXTERNAL_ICEBERG.TRANSACTIONS
GROUP BY 1, 2
ORDER BY txn_count DESC;

In [ ]:
-- Query VARIANT in External PRODUCTS
SELECT 
    category,
    attributes:color::STRING AS color,
    COUNT(*) AS product_count,
    AVG(price) AS avg_price,
    AVG(attributes:weight::FLOAT) AS avg_weight
FROM ICEBERG_POC.EXTERNAL_ICEBERG.PRODUCTS
GROUP BY 1, 2
ORDER BY product_count DESC;

---
## Test 7: Aggregations & Window Functions

In [ ]:
-- Window functions on External TRANSACTIONS
SELECT 
    customer_id,
    transaction_type,
    transaction_ts,
    amount,
    SUM(amount) OVER (PARTITION BY customer_id ORDER BY transaction_ts) AS running_total,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY transaction_ts) AS txn_sequence,
    LAG(amount) OVER (PARTITION BY customer_id ORDER BY transaction_ts) AS prev_amount
FROM ICEBERG_POC.EXTERNAL_ICEBERG.TRANSACTIONS
WHERE customer_id IN (100, 200, 300)
ORDER BY customer_id, transaction_ts
LIMIT 50;

---
## Test 8: Query Performance Analysis - All Table Types

In [ ]:
-- Compare performance across all table types
SELECT 
    CASE 
        WHEN QUERY_TEXT ILIKE '%NATIVE_BASELINE%' THEN 'Native'
        WHEN QUERY_TEXT ILIKE '%EXTERNAL_ICEBERG%' THEN 'Iceberg (External)'
        WHEN QUERY_TEXT ILIKE '%TESTS.%' THEN 'Iceberg (Internal)'
        ELSE 'Other'
    END AS table_type,
    COUNT(*) AS query_count,
    ROUND(AVG(TOTAL_ELAPSED_TIME) / 1000, 3) AS avg_elapsed_sec,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000, 3) AS p50_sec,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000, 3) AS p95_sec,
    ROUND(AVG(BYTES_SCANNED) / 1024 / 1024, 2) AS avg_mb_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION())
WHERE QUERY_TYPE = 'SELECT'
    AND EXECUTION_STATUS = 'SUCCESS'
    AND (QUERY_TEXT ILIKE '%NATIVE_BASELINE%' 
         OR QUERY_TEXT ILIKE '%EXTERNAL_ICEBERG%' 
         OR QUERY_TEXT ILIKE '%TESTS.EVENTS%')
GROUP BY 1
ORDER BY 1;

---
## Test 9: Warehouse Size Scaling

In [ ]:
-- XS Warehouse - External Iceberg
USE WAREHOUSE POC_WH_XS;

SELECT 'XS - External' AS test, COUNT(*), SUM(amount), AVG(amount)
FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS
WHERE region = 'US-EAST';

In [ ]:
-- Medium Warehouse - External Iceberg
USE WAREHOUSE POC_WH_M;

SELECT 'M - External' AS test, COUNT(*), SUM(amount), AVG(amount)
FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS
WHERE region = 'US-EAST';

In [ ]:
-- Large Warehouse - External Iceberg
USE WAREHOUSE POC_WH_L;

SELECT 'L - External' AS test, COUNT(*), SUM(amount), AVG(amount)
FROM ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS
WHERE region = 'US-EAST';

In [ ]:
-- Reset warehouse
USE WAREHOUSE COMPUTE_WH;

---
## Test 10: External Storage Metadata Paths

In [ ]:
-- Verify external storage paths for interoperability
SELECT 
    'EVENTS' AS table_name,
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.EVENTS')):metadataLocation::STRING AS metadata_path
UNION ALL
SELECT 'CUSTOMERS', PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.CUSTOMERS')):metadataLocation::STRING
UNION ALL
SELECT 'ORDERS', PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.ORDERS')):metadataLocation::STRING
UNION ALL
SELECT 'PRODUCTS', PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.PRODUCTS')):metadataLocation::STRING
UNION ALL
SELECT 'TRANSACTIONS', PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.TRANSACTIONS')):metadataLocation::STRING;

---
## Summary: Performance Results

### Table Comparison
| Test | Native | Iceberg (Internal) | Iceberg (External) | Status |
|------|--------|--------------------|--------------------|--------|
| Full Scan (1M rows) | TBD | TBD | TBD | ⏳ |
| Point Lookup | TBD | TBD | TBD | ⏳ |
| Multi-Join | TBD | TBD | TBD | ⏳ |
| VARIANT Query | TBD | TBD | TBD | ⏳ |

### External Iceberg Tables
| Table | Rows | Storage Path |
|-------|------|-------------|
| EVENTS | 1M | `external_iceberg/events/` |
| CUSTOMERS | 100K | `external_iceberg/customers/` |
| ORDERS | 500K | `external_iceberg/orders/` |
| PRODUCTS | 10K | `external_iceberg/products/` |
| TRANSACTIONS | 1M | `external_iceberg/transactions/` |

*Run query history analysis cells to populate actual performance metrics*